<a href="https://colab.research.google.com/github/divya-dataengineer/data-engineering-practice/blob/main/Data_Quality_Checks_In_PySpark.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, trim, lower, upper, regexp_extract,
    to_date, expr, when, count, sum as spark_sum,
    current_date, row_number, concat_ws, lit
)
from pyspark.sql.window import Window

spark = SparkSession.builder \
    .appName("OrdersDataQuality") \
    .getOrCreate()

data = [
    (101, "C001", "12-Oct-26", 500.0, 2, "a@gmail.com", "Completed", "India"),
    (102, "C002", "13/10/2026", 250.0, 1, "b@gmail.com", "Pending", "India"),
    (103, "C003", "2026-10-14", -50.0, 1, "bad-email", "Completed", "India"),
    (104, None, "invalid-date", 100.0, 2, "c@gmail.com", "Shipped", "India"),
    (105, "C005", None, None, 0, "d@gmail.com", "Unknown", None),
    (105, "C005", None, None, 0, "d@gmail.com", "Unknown", None),
    (None, "C006", "01-Jan-27", 100.0, 3, "e@gmail.com", "Pending", "India"),
    (107, "C007", "2026-10-10", 0.0, -2, "f@gmail.com", None, "USA"),
    (108, "C008", "15/10/2026", 200.0, 1, " ", "Completed", "India"),
    (109, "C009", "2026-10-11", 150.0, 2, "i@gmail.com", "Completed", "India")
]

columns = [
    "OrderID", "CustomerID", "OrderDate_Raw", "Amount",
    "Quantity", "Email", "Status", "Country"
]

orders_df = spark.createDataFrame(data, columns)

orders_df.show(truncate=False)

+-------+----------+-------------+------+--------+-----------+---------+-------+
|OrderID|CustomerID|OrderDate_Raw|Amount|Quantity|Email      |Status   |Country|
+-------+----------+-------------+------+--------+-----------+---------+-------+
|101    |C001      |12-Oct-26    |500.0 |2       |a@gmail.com|Completed|India  |
|102    |C002      |13/10/2026   |250.0 |1       |b@gmail.com|Pending  |India  |
|103    |C003      |2026-10-14   |-50.0 |1       |bad-email  |Completed|India  |
|104    |NULL      |invalid-date |100.0 |2       |c@gmail.com|Shipped  |India  |
|105    |C005      |NULL         |NULL  |0       |d@gmail.com|Unknown  |NULL   |
|105    |C005      |NULL         |NULL  |0       |d@gmail.com|Unknown  |NULL   |
|NULL   |C006      |01-Jan-27    |100.0 |3       |e@gmail.com|Pending  |India  |
|107    |C007      |2026-10-10   |0.0   |-2      |f@gmail.com|NULL     |USA    |
|108    |C008      |15/10/2026   |200.0 |1       |           |Completed|India  |
|109    |C009      |2026-10-

In [3]:
orders_df = orders_df.withColumn(
    "OrderDate",
    expr("""
        coalesce(
            try_to_timestamp(trim(OrderDate_Raw), 'd-MMM-yy'),
            try_to_timestamp(trim(OrderDate_Raw), 'M/d/yyyy'),
            try_to_timestamp(trim(OrderDate_Raw), 'yyyy-MM-dd')
        )
    """).cast("date")
)

orders_df.show()

+-------+----------+-------------+------+--------+-----------+---------+-------+----------+
|OrderID|CustomerID|OrderDate_Raw|Amount|Quantity|      Email|   Status|Country| OrderDate|
+-------+----------+-------------+------+--------+-----------+---------+-------+----------+
|    101|      C001|    12-Oct-26| 500.0|       2|a@gmail.com|Completed|  India|2026-10-12|
|    102|      C002|   13/10/2026| 250.0|       1|b@gmail.com|  Pending|  India|      NULL|
|    103|      C003|   2026-10-14| -50.0|       1|  bad-email|Completed|  India|2026-10-14|
|    104|      NULL| invalid-date| 100.0|       2|c@gmail.com|  Shipped|  India|      NULL|
|    105|      C005|         NULL|  NULL|       0|d@gmail.com|  Unknown|   NULL|      NULL|
|    105|      C005|         NULL|  NULL|       0|d@gmail.com|  Unknown|   NULL|      NULL|
|   NULL|      C006|    01-Jan-27| 100.0|       3|e@gmail.com|  Pending|  India|2027-01-01|
|    107|      C007|   2026-10-10|   0.0|      -2|f@gmail.com|     NULL|    USA|

In [5]:
order_id_window = Window.partitionBy("OrderID")

orders_df = orders_df.withColumn(
    "OrderID_Count",
    count("*").over(order_id_window)
)

orders_df = orders_df.withColumn(
    "DQ_Duplicate_OrderID",
    when(
        col("OrderID").isNotNull() &
        (col("OrderID_Count") > 1),
        "FAIL"
    ).otherwise("PASS")
)

orders_df.show()

+-------+----------+-------------+------+--------+-----------+---------+-------+----------+-------------+--------------------+
|OrderID|CustomerID|OrderDate_Raw|Amount|Quantity|      Email|   Status|Country| OrderDate|OrderID_Count|DQ_Duplicate_OrderID|
+-------+----------+-------------+------+--------+-----------+---------+-------+----------+-------------+--------------------+
|   NULL|      C006|    01-Jan-27| 100.0|       3|e@gmail.com|  Pending|  India|2027-01-01|            1|                PASS|
|    101|      C001|    12-Oct-26| 500.0|       2|a@gmail.com|Completed|  India|2026-10-12|            1|                PASS|
|    102|      C002|   13/10/2026| 250.0|       1|b@gmail.com|  Pending|  India|      NULL|            1|                PASS|
|    103|      C003|   2026-10-14| -50.0|       1|  bad-email|Completed|  India|2026-10-14|            1|                PASS|
|    104|      NULL| invalid-date| 100.0|       2|c@gmail.com|  Shipped|  India|      NULL|            1|      

In [6]:
# 1. OrderID: required
orders_df = orders_df.withColumn(
    "DQ_OrderID",
    when(col("OrderID").isNull(), "FAIL").otherwise("PASS")
)

# 2. CustomerID: required and non-blank
orders_df = orders_df.withColumn(
    "DQ_CustomerID",
    when(
        col("CustomerID").isNull() |
        (trim(col("CustomerID")) == ""),
        "FAIL"
    ).otherwise("PASS")
)

# 3. OrderDate: required and successfully parsed
orders_df = orders_df.withColumn(
    "DQ_OrderDate",
    when(
        col("OrderDate").isNull(),
        "FAIL"
    ).otherwise("PASS")
)

# 4. OrderDate: must not be in the future
orders_df = orders_df.withColumn(
    "DQ_FutureDate",
    when(
        col("OrderDate").isNotNull() &
        (col("OrderDate") > current_date()),
        "FAIL"
    ).otherwise("PASS")
)

# 5. Amount: required and must be positive
orders_df = orders_df.withColumn(
    "DQ_Amount",
    when(
        col("Amount").isNull() | (col("Amount") <= 0),
        "FAIL"
    ).otherwise("PASS")
)

# 6. Quantity: required and must be a positive whole number
orders_df = orders_df.withColumn(
    "DQ_Quantity",
    when(
        col("Quantity").isNull() |
        (col("Quantity") <= 0) |
        (col("Quantity") % 1 != 0),
        "FAIL"
    ).otherwise("PASS")
)

# 7. Email: basic format validation
orders_df = orders_df.withColumn(
    "DQ_Email",
    when(
        col("Email").isNull() |
        (trim(col("Email")) == "") |
        (~lower(trim(col("Email"))).rlike(
            r"^[a-z0-9._%+-]+@[a-z0-9.-]+\.[a-z]{2,}$"
        )),
        "FAIL"
    ).otherwise("PASS")
)

# 8. Status: allowed values only
allowed_statuses = ["COMPLETED", "PENDING", "SHIPPED", "CANCELLED"]

orders_df = orders_df.withColumn(
    "DQ_Status",
    when(
        col("Status").isNull() |
        (~upper(trim(col("Status"))).isin(allowed_statuses)),
        "FAIL"
    ).otherwise("PASS")
)

# 9. Country: required and from an allowed list
allowed_countries = ["INDIA", "USA", "UK", "CANADA"]

orders_df = orders_df.withColumn(
    "DQ_Country",
    when(
        col("Country").isNull() |
        (trim(col("Country")) == "") |
        (~upper(trim(col("Country"))).isin(allowed_countries)),
        "FAIL"
    ).otherwise("PASS")
)

orders_df.select(
    "OrderID",
    "CustomerID",
    "OrderDate_Raw",
    "OrderDate",
    "Amount",
    "Quantity",
    "Email",
    "Status",
    "Country",
    "DQ_Duplicate_OrderID",
    "DQ_OrderID",
    "DQ_CustomerID",
    "DQ_OrderDate",
    "DQ_FutureDate",
    "DQ_Amount",
    "DQ_Quantity",
    "DQ_Email",
    "DQ_Status",
    "DQ_Country"
).show(truncate=False)

+-------+----------+-------------+----------+------+--------+-----------+---------+-------+--------------------+----------+-------------+------------+-------------+---------+-----------+--------+---------+----------+
|OrderID|CustomerID|OrderDate_Raw|OrderDate |Amount|Quantity|Email      |Status   |Country|DQ_Duplicate_OrderID|DQ_OrderID|DQ_CustomerID|DQ_OrderDate|DQ_FutureDate|DQ_Amount|DQ_Quantity|DQ_Email|DQ_Status|DQ_Country|
+-------+----------+-------------+----------+------+--------+-----------+---------+-------+--------------------+----------+-------------+------------+-------------+---------+-----------+--------+---------+----------+
|NULL   |C006      |01-Jan-27    |2027-01-01|100.0 |3       |e@gmail.com|Pending  |India  |PASS                |FAIL      |PASS         |PASS        |FAIL         |PASS     |PASS       |PASS    |PASS     |PASS      |
|101    |C001      |12-Oct-26    |2026-10-12|500.0 |2       |a@gmail.com|Completed|India  |PASS                |PASS      |PASS     